In [ ]:
import os
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#### Functions

In [ ]:
# plot data types
def plot_data_types(df_data, str_filename='output/data_types.png'):
    # get value counts of dtypes
    ser_dtype_counts = df_data.dtypes.astype(str).value_counts()
    # plot
    fig, ax = plt.subplots(figsize=(10, 5))
    ser_dtype_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title('Data Types', fontsize=16)
    ax.set_xlabel('Data Type', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    # annotate bars
    for i, int_val in enumerate(ser_dtype_counts):
        ax.text(i, int_val + 0.1, str(int_val), ha='center', fontsize=11)
    plt.tight_layout()
    plt.savefig(str_filename, dpi=300)
    plt.show()

# plot proportion missing
def plot_proportion_missing(df_data, str_filename='output/proportion_missing.png'):
    # calculate proportion missing
    ser_missing = df_data.isnull().mean()
    # filter to columns with missing values
    ser_missing = ser_missing[ser_missing > 0].sort_values(ascending=True)
    # if no missing values
    if len(ser_missing) == 0:
        print('No missing values found.')
        return
    # plot
    fig, ax = plt.subplots(figsize=(10, max(5, len(ser_missing) * 0.4)))
    ser_missing.plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title('Proportion Missing by Column', fontsize=16)
    ax.set_xlabel('Proportion Missing', fontsize=12)
    ax.set_ylabel('Column', fontsize=12)
    plt.tight_layout()
    plt.savefig(str_filename, dpi=300)
    plt.show()

# plot target distribution
def plot_target(df_data, str_target, str_filename='output/target_distribution.png'):
    # value counts
    ser_counts = df_data[str_target].value_counts()
    ser_proportions = df_data[str_target].value_counts(normalize=True)
    # plot
    fig, ax = plt.subplots(figsize=(8, 5))
    ser_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title('Target Distribution', fontsize=16)
    ax.set_xlabel(str_target, fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    # annotate bars with count and proportion
    for i, (int_count, flt_prop) in enumerate(zip(ser_counts, ser_proportions)):
        ax.text(i, int_count + (ser_counts.max() * 0.01), f'{int_count:,}\n({flt_prop:.2%})', ha='center', fontsize=11)
    plt.tight_layout()
    plt.savefig(str_filename, dpi=300)
    plt.show()

# descriptive statistics
def descriptive_statistics(df_data):
    df_desc = df_data.describe().T.reset_index()
    df_desc = df_desc.rename(columns={'index': 'str_column'})
    return df_desc

# plot violin plots
def plot_violin(df_data, str_target, str_filename='output/violin_plots.png'):
    # get numeric columns excluding target
    list_numeric_cols = [col for col in df_data.select_dtypes(include=[np.number]).columns if col != str_target]
    int_n_cols = 3
    int_n_rows = int(np.ceil(len(list_numeric_cols) / int_n_cols))
    fig, axes = plt.subplots(int_n_rows, int_n_cols, figsize=(6 * int_n_cols, 4 * int_n_rows))
    axes = axes.flatten() if int_n_rows > 1 else (axes if hasattr(axes, '__iter__') else [axes])
    for i, str_col in enumerate(list_numeric_cols):
        sns.violinplot(data=df_data, x=str_target, y=str_col, ax=axes[i], palette='muted')
        axes[i].set_title(str_col, fontsize=12)
        axes[i].set_xlabel('')
        axes[i].set_ylabel('')
    # hide unused axes
    for j in range(len(list_numeric_cols), len(axes)):
        axes[j].set_visible(False)
    plt.suptitle('Violin Plots by Target', fontsize=16, y=1.01)
    plt.tight_layout()
    plt.savefig(str_filename, dpi=300, bbox_inches='tight')
    plt.show()

# plot correlation with target
def plot_correlation_with_target(df_data, str_target, str_filename='output/correlation_with_target.png'):
    # compute correlations with target
    ser_corr = df_data.select_dtypes(include=[np.number]).corr()[str_target].drop(str_target)
    # sort by absolute value
    ser_corr = ser_corr.reindex(ser_corr.abs().sort_values(ascending=True).index)
    # plot
    fig, ax = plt.subplots(figsize=(10, max(5, len(ser_corr) * 0.4)))
    list_colors = ['steelblue' if x >= 0 else 'salmon' for x in ser_corr]
    ser_corr.plot(kind='barh', ax=ax, color=list_colors, edgecolor='black')
    ax.set_title('Correlation with Target', fontsize=16)
    ax.set_xlabel('Correlation', fontsize=12)
    ax.set_ylabel('Feature', fontsize=12)
    ax.axvline(x=0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.savefig(str_filename, dpi=300)
    plt.show()

#### Constants

In [ ]:
# bucket
str_bucket = os.getcwd().split('/')[4].replace('_', '-')
print(f'Bucket: {str_bucket}')

# step
str_step = os.getcwd().split('/')[-1]
print(f'Step: {str_step}')

# s3 path
str_s3_path = f's3://{str_bucket}/00_data_collection/data.csv'
print(f'S3 Path: {str_s3_path}')

# target
str_target = 'charged_off'
print(f'Target: {str_target}')

# output directory
os.makedirs('output', exist_ok=True)

#### Read Data

In [ ]:
# read data
df_raw = pd.read_csv(str_s3_path)

# rows and columns
int_n_rows = df_raw.shape[0]
int_n_cols = df_raw.shape[1]
print(f'Rows: {int_n_rows:,}')
print(f'Columns: {int_n_cols}')

# date range (if date column exists)
list_date_cols = df_raw.select_dtypes(include=['datetime64']).columns.tolist()
if len(list_date_cols) == 0:
    # try to find date-like object columns
    for str_col in df_raw.select_dtypes(include=['object']).columns:
        try:
            pd.to_datetime(df_raw[str_col].head(100))
            list_date_cols.append(str_col)
        except (ValueError, TypeError):
            pass
if len(list_date_cols) > 0:
    str_date_col = list_date_cols[0]
    df_raw[str_date_col] = pd.to_datetime(df_raw[str_date_col])
    print(f'Date Column: {str_date_col}')
    print(f'Date Range: {df_raw[str_date_col].min()} to {df_raw[str_date_col].max()}')

# target mean
flt_target_mean = df_raw[str_target].mean()
print(f'Target Mean: {flt_target_mean:.4f}')

# preview
df_raw.head()

#### Data Types

In [ ]:
plot_data_types(df_raw)

#### Missing Values

In [ ]:
plot_proportion_missing(df_raw)

#### Target Variable

In [ ]:
plot_target(df_raw, str_target)

#### Descriptive Statistics

In [ ]:
df_desc = descriptive_statistics(df_raw)
df_desc.to_csv('output/descriptive_statistics.csv', index=False)
df_desc

#### Violin Plots

In [ ]:
plot_violin(df_raw, str_target)

#### Correlation with Target

In [ ]:
plot_correlation_with_target(df_raw, str_target)

#### Save

In [ ]:
# save to s3
str_s3_output = f's3://{str_bucket}/{str_step}/data.csv'
df_raw.to_csv(str_s3_output, index=False)
print(f'Data saved to {str_s3_output}')